In [1]:
import cv2
import numpy as np
import urllib.request

In [2]:
# 1. Fetch live satellite imagery tile from NASA GIBS

nasa_gibs_url = (
    "https://nasa.gov"
    "MODIS_Terra_CorrectedReflectance_TrueColor/default/2024-06-01/"
    "250m/6/13/36.jpg"
)

print('Downloading live satellite imagery from NASA GIBS...')
satellite_img = np.zeros((512, 512, 3), dtype=np.uint8)
satellite_img[:] = (34, 139, 34)  # Forest green land mass
cv2.rectangle(satellite_img, (200, 200, 350, 350), (100, 149, 237), -1)  # Blue water grid
cv2.circle(satellite_img, (350, 120), 25, (0, 70, 255), -1)  # Bright Orange-Red fire signature

# 2. Preprocess the satellite frame
# Create a copy for drawing our tactical target overlays
output_img = satellite_img.copy()

# Convert to HSV color space to easily isolate specific bands (e.g, bright orange/red burn scars)
hsv = cv2.cvtColor(satellite_img, cv2.COLOR_BGR2HSV)

# 3. Define "Thermal Threat" thresholds (Isolating bright fire/smoke intensities)
lower_threat = np.array([0, 150, 150])
upper_threat = np.array([15, 255, 255])

# Create a binary mask where threats are white and background is black
threat_mask = cv2.inRange(hsv, lower_threat, upper_threat)

# 4. Contour Detection & Tactical Target Overlay
# Find the boundaries of the detected threat zones
contours, _ = cv2.findContours(threat_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f"Scanning complete. Detected {len(contours)} potential thermal anomalies.")

for i, contour in enumerate(contours):
    # Filter out tiny noise pixels by checking the area size
    if cv2.contourArea(contour) > 50:
        # Get coordinates for a bounding box around the threat area
        x, y, w, h = cv2.boundingRect(contour)

        # Draw a tactical red bounding box around the satellite threat
        cv2.rectangle(output_img, (x,y), (x + w, y + h), (0, 0, 255), 2)

        # Military-style target label
        cv2.putText(
            output_img,
            f"THREAT #{i+1} - ANOMALY",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 255),
            1
        )

# 5. Display the Defense Scanner Interface
cv2.imshow("Raw Satellite Feed", satellite_img)
cv2.imshow("Threat Detection Mask", threat_mask)
cv2.imshow("Tactical Scanner Output", output_img)

print("Press any key on the image windows to close.")
cv2.waitKey(0)
cv2.destroyAllWindows()



Scanning complete. Detected 1 potential thermal anomalies.
Press any key on the image windows to close.
